In [ ]:
%load_ext autoreload
%autoreload 2
%autosave 30
%matplotlib inline

# Comparison and validation of FID implementation
As we used an autoencoder for the FID computation this notebook explores if this method can actually give insighs to the realness of the post-processed forecasts

For this, we will:
- Compute the FID for different post prosessing models
  - and explore relationships to the crps and other error measures
- Use different specification of the autoencoder
  - to explore how the size of the hidden dimension influences results
- Test if the FID increases when non-realistic forecasts are presented
  - for this we can use the averaged forecasts of the ensemble which should be "too smooth"
  - inject some sort of noise to the forecasts

In [ ]:
import glob
from itertools import product

import hvplot.polars  # noqa: F401
import polars as pl
import torch

from genpp import BASE_DIR
from genpp.eval.FID import fid

First read all precomputed means and standardeviations for different post processing models.\
These are saved in the folder `FID/save_latent`.\
For now all latent vectors are computed using this autoencoder: https://wandb.ai/feik/genpp/sweeps/3mnyq2ej/runs/nyuw6ut1

In [ ]:
storage_dir = BASE_DIR / "eval" / "FID" / "save_latent"
precomputed_latents_files = glob.glob(str(storage_dir / "*.pt"))

# Put them all in a large dict and unwrapt the inner dict if there is
latents_dict = {}
for f in precomputed_latents_files:
    d = torch.load(f)
    for item in d:
        latents_dict[item["model"]] = {"mean": item["mean"], "cov": item["cov"]}

In [ ]:
# Get the ground truth latents (these are all that contain autoencoder in their name)
gt_latents = {k: v for k, v in latents_dict.items() if "autoencoder" in k}

# Now compute FID for all other models against the ground truth ones
rows = []
for (gt_name, gt_model_latents), (model_name, model_latents) in product(
    gt_latents.items(), latents_dict.items()
):
    fid_value = fid(
        mu1=gt_model_latents["mean"],
        sigma1=gt_model_latents["cov"],
        mu2=model_latents["mean"],
        sigma2=model_latents["cov"],
    )
    rows.append(
        {
            "gt_model": gt_name,
            "model": model_name,
            "fid": fid_value,
        }
    )

# Clip values to be non-negative as these come from numerical errors
fid_df = pl.DataFrame(rows).with_columns(pl.col("fid").clip(0, None))
fid_df

## Notes on the following Plot

In the following we will compare the fid scores of the postprocessing models.\
Note that for all postprocessing models, the latents stem from the **validation set**.\
So the bad score for the other sets are somewhat expected.\
However the "real" forecasts should never have a worse score than the post-processed ones. This checks out and is a good sign.\


In [ ]:
fid_df.hvplot.heatmap(  # pyright: ignore[reportAttributeAccessIssue]
    x="gt_model",
    y="model",
    C="fid",
    cmap="Reds",
    title="FID Scores Between Models",
    width=800,
    height=600,
    colorbar=True,
)

## Lets add some intentionlly "bad" forecasts
For this we use the validation set of real forecasts and apply some transformations such as:
- gaussian noise
- gaussian blur
- black rectangles
- swirl
- salt and pepper noise

This list is taken from the FID paper: [GANs Trained by a Two Time-Scale Update Rule Converge to a Local Nash Equilibrium](https://arxiv.org/abs/1706.08500) (Heusel et al., 2018).

But first lets start with the mean forecast

In [ ]:
import lightning as L

from genpp.eval.FID.encoder import AutoEncoder

In [ ]:
# Lets load the model
RUN_PATH_AE = "feik/genpp/nyuw6ut1"
MODEL_ID_AE = RUN_PATH_AE.split("/")[-1]
OUTPUT_DIR_AE = BASE_DIR.parent.parent / "outputs"

MODEL_DIR_AE = list(OUTPUT_DIR_AE.rglob(f"*{MODEL_ID_AE}*"))[0].parent.parent.parent
MODEL_CHECKPOINT_AE = list(MODEL_DIR_AE.rglob("*.ckpt"))[0]

ae = AutoEncoder.load_from_checkpoint(MODEL_CHECKPOINT_AE)

In [ ]:
# Load the data
import xarray as xr
from torch.utils.data import DataLoader, TensorDataset

from genpp.data import FC_VARS

ens_mean = xr.open_dataset(BASE_DIR / "data" / "weatherbench2" / "ens_flat_agg.zarr")
val_slice = slice("2021-01-01", "2021-12-31")

ens_mean = (
    ens_mean.sel(statistic="mean", time=val_slice)
    .stack(prediction=["time", "prediction_timedelta"])[FC_VARS]
    .to_dataarray("feature")
    .transpose("prediction", "feature", "longitude", "latitude")
)
mean_fc_tensor = torch.tensor(ens_mean.values).float()
mean_fc_dataset = TensorDataset(mean_fc_tensor)
mean_fc_dataloader = DataLoader(mean_fc_dataset, batch_size=128, shuffle=False)

In [ ]:
trainer = L.Trainer(logger=False)
predictions = trainer.predict(ae, mean_fc_dataloader)
predictions = torch.cat(predictions, dim=0)  # type: ignore

means = predictions.mean(dim=0)
cov = torch.cov(predictions.T)

In [ ]:
for gt_name, gt_model_latents in gt_latents.items():
    fid_value = fid(
        mu1=gt_model_latents["mean"],
        sigma1=gt_model_latents["cov"],
        features2=predictions,  # type: ignore
    )
    rows.append(
        {
            "gt_model": gt_name,
            "model": "mean_fc",
            "fid": fid_value,
        }
    )
# Clip values to be non-negative as these come from numerical errors
fid_df = pl.DataFrame(rows).with_columns(pl.col("fid").clip(0, None))
fid_df

In [ ]:
fid_df.hvplot.heatmap(  # pyright: ignore[reportAttributeAccessIssue]
    x="gt_model",
    y="model",
    C="fid",
    cmap="Reds",
    title="FID Scores Between Models",
    width=800,
    height=600,
    colorbar=True,
)

### Few notes on this

If the Autoencoder is good at encoding forecasts it is probably also good at encoding smoothed forecasts.\
Instead of using an autoencoder use some classifier that can differentiate between forecasts and too smooth forecasts (i.e. mean forecasts)


### Now add some intentional noise

In [ ]:
from torchvision.transforms import v2

from genpp.data.zarr_dataset import TransformTensorDataset

In [ ]:
ens_fc = (
    xr.open_dataset(BASE_DIR / "data" / "weatherbench2" / "ifs_ens.zarr")
    .sel(time=val_slice)[FC_VARS]
    .stack(prediction=["time", "prediction_timedelta", "number"])
    .to_dataarray("feature")
    .transpose("prediction", "feature", "longitude", "latitude")
)

In [ ]:
gaussian_noise = v2.GaussianNoise(mean=0.0, sigma=1, clip=False)  # p=1 by default
gaussian_blur = v2.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 2.0))  # p=1 by default
black_rectangle = v2.RandomErasing(
    p=1,
    scale=(0.02, 0.33),
    ratio=(0.3, 3.3),
    value=tuple(ens_fc.mean(["prediction", "longitude", "latitude"]).values),  # type: ignore
)

defects = [gaussian_noise, gaussian_blur, black_rectangle]

for defect in defects:
    name = "Real_" + defect.__class__.__name__
    ens_fc_ds = TransformTensorDataset(torch.tensor(ens_fc.values).float(), transform=defect)
    dl = DataLoader(ens_fc_ds, batch_size=265, shuffle=False)
    predictions = trainer.predict(ae, dl)
    predictions = torch.cat(predictions, dim=0)  # type: ignore
    for gt_name, gt_model_latents in gt_latents.items():
        fid_value = fid(
            mu1=gt_model_latents["mean"],
            sigma1=gt_model_latents["cov"],
            features2=predictions,  # type: ignore
        )
        rows.append(
            {
                "gt_model": gt_name,
                "model": name,
                "fid": fid_value,
            }
        )

fid_df = pl.DataFrame(rows).with_columns(pl.col("fid").clip(0, None))
fid_df

In [ ]:
fid_df.hvplot.heatmap(  # pyright: ignore[reportAttributeAccessIssue]
    x="gt_model",
    y="model",
    C="fid",
    cmap="Reds",
    title="FID Scores Between Models",
    width=800,
    height=600,
    colorbar=True,
)